# Notebook to fix PR issues


### Generate the UE data from mobility model first

In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
import scipy
import numpy as np
from radp_library import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from radp.digital_twin.mobility.param_regression import get_predicted_alpha,preprocess_ue_data
from radp.digital_twin.utils.cell_selection import perform_attachment
from radp.digital_twin.rf.bayesian.bayesian_engine import (
    BayesianDigitalTwin,
    NormMethod,
)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [3]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [4]:
training_data = get_ue_data(params)
training_data.head()

,mock_ue_id,lon,lat,tick
0,0,48.510339,-16.462645,0
1,1,19.286613,63.617111,0
2,2,21.315702,-47.889952,0
3,3,-70.559364,-79.511709,0
4,4,-168.916011,-39.338640,0


In [3]:
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [3]:
def _prepare_all_UEs_from_all_cells_df(
     data, topology
    ) -> pd.DataFrame:
        """
        Connects each user equipment (UE) entry to all cells in the topology for each tick,
        effectively creating a Cartesian product of UEs and cells, which includes data from both sources.
        """

        ue_data = data
        ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})
        topology_tmp = topology
        
        """" 
        As 'ue_data' and 'topology_tmp' both have columns 'cell_id', we encounter a name conflict when merging 
        with column 'Key' (merge key is not set to cell_id). In order to resolve this, pandas adds a suffix to 
        the conflicting columns as 'cell_id_x' and 'cell_id_y'  
        """  
        if 'cell_id' in ue_data.columns:
            ue_data = ue_data.drop(columns = ['cell_id'])
        
        # Remove the 'cell_' prefix and convert cell_id to integer if needed
        if topology_tmp["cell_id"].dtype == object:
            topology_tmp["cell_id"] = (
                topology_tmp["cell_id"].str.replace("cell_", "").astype(int)
            )
        
        ue_data["key"] = 1
        topology_tmp["key"] = 1
        combined_df = pd.merge(ue_data, topology_tmp, on="key").drop("key", axis=1)
        print("Combined DataFrame:", combined_df)
        return combined_df


In [4]:
def _preprocess_ue_topology_data(data,topology) -> pd.DataFrame:
        full_data = _prepare_all_UEs_from_all_cells_df(data,topology)
        full_data["log_distance"] = full_data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        full_data["cell_rxpwr_dbm"] = full_data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        return full_data

In [5]:
def _preprocess_ue_training_data(data,topology) -> pd.DataFrame:
        data = _preprocess_ue_topology_data(data,topology)
        train_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
        desired_idxs = [1 + r for r in range(n_cell)]

        n_samples_train = []
        for df in train_per_cell_df:
            n_samples_train.append(df.shape[0])

        train_per_cell_df_processed = []
        for i in range(n_cell):
            train_per_cell_df_processed.append(
                get_percell_data(
                    data_in=train_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_train[i],
                )[0][0]
            )

        training_data = {}

        for i, df in enumerate(train_per_cell_df_processed):
            train_cell_id = idx_cell_id_mapping[i + 1]
            training_data[train_cell_id] = df

        for train_cell_id, training_data_idx in training_data.items():
            training_data_idx["cell_id"] = train_cell_id
            training_data_idx["cell_lat"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lat"].values[0]
            training_data_idx["cell_lon"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lon"].values[0]
            training_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_az_deg"].values[0]
            training_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            training_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    training_data_idx["cell_az_deg"].values[0],
                    training_data_idx["cell_lat"].values[0],
                    training_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    training_data_idx["latitude"], training_data_idx["longitude"]
                )
            ]

        return training_data

In [6]:
bayesian_digital_twins = {}

In [6]:
def _training(bayesian_digital_twins, maxiter: int, train_data: pd.DataFrame,topology: pd.DataFrame) -> List[float]:
        """
        Trains the Bayesian Digital Twins for each cell in the topology using the UE locations and features
        like log distance, relative bearing, and cell received power (Rx power).
        """
        training_data = _preprocess_ue_training_data(train_data,topology)
        loss_vs_iters = []
        for train_cell_id, training_data_idx in training_data.items():
            bayesian_digital_twins[train_cell_id] = BayesianDigitalTwin(
                data_in=[training_data_idx],
                x_columns=["log_distance", "relative_bearing"],
                y_columns=["cell_rxpwr_dbm"],
                norm_method=NormMethod.MINMAX,
            )
            bayesian_digital_twins[train_cell_id] = bayesian_digital_twins[
                train_cell_id
            ]
            loss_vs_iters.append(
                bayesian_digital_twins[train_cell_id].train_distributed_gpmodel(
                    maxiter=maxiter,
                )
            )
        return bayesian_digital_twins, loss_vs_iters

In [13]:
bayesian_digital_twins, loss_vs_iters = _training(
    bayesian_digital_twins,
    maxiter=100,
    train_data=training_data,
    topology=topology,
)


Combined DataFrame:       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon  cell_id  \
0              0   48.510339 -16.462645     0     -90.0    -180.0        1   
1              0   48.510339 -16.462645     0       0.0       0.0        2   
2              0   48.510339 -16.462645     0      90.0     180.0        3   
3              1   19.286613  63.617111     0     -90.0    -180.0        1   
4              1   19.286613  63.617111     0       0.0       0.0        2   
...          ...         ...        ...   ...       ...       ...      ...   
4795          30   89.100351 -11.651079    49       0.0       0.0        2   
4796          30   89.100351 -11.651079    49      90.0     180.0        3   
4797          31  146.478589  74.187621    49     -90.0    -180.0        1   
4798          31  146.478589  74.187621    49       0.0       0.0        2   
4799          31  146.478589  74.187621    49      90.0     180.0        3   

      cell_az_deg  cell_carrier_freq_mhz  


[2025-05-07 14:45:46,036] INFO:  Iter 1/100 - Loss: 0.770 (delta=inf)
[2025-05-07 14:45:46,070] INFO:  Iter 2/100 - Loss: 0.750 (delta=-0.019613)
[2025-05-07 14:45:46,111] INFO:  Iter 3/100 - Loss: 0.731 (delta=-0.018802)
[2025-05-07 14:45:46,145] INFO:  Iter 4/100 - Loss: 0.713 (delta=-0.017859)
[2025-05-07 14:45:46,174] INFO:  Iter 5/100 - Loss: 0.694 (delta=-0.019280)
[2025-05-07 14:45:46,209] INFO:  Iter 6/100 - Loss: 0.675 (delta=-0.019566)
[2025-05-07 14:45:46,239] INFO:  Iter 7/100 - Loss: 0.658 (delta=-0.016846)
[2025-05-07 14:45:46,271] INFO:  Iter 8/100 - Loss: 0.639 (delta=-0.018843)
[2025-05-07 14:45:46,304] INFO:  Iter 9/100 - Loss: 0.617 (delta=-0.021821)
[2025-05-07 14:45:46,338] INFO:  Iter 10/100 - Loss: 0.597 (delta=-0.019510)
[2025-05-07 14:45:46,371] INFO:  Iter 11/100 - Loss: 0.575 (delta=-0.022858)
[2025-05-07 14:45:46,403] INFO:  Iter 12/100 - Loss: 0.555 (delta=-0.019504)
[2025-05-07 14:45:46,435] INFO:  Iter 13/100 - Loss: 0.535 (delta=-0.020242)
[2025-05-07 14

In [14]:
bayesian_digital_twins

{1: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x24dd9f45d20>,
 2: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x24dbbf6b730>,
 3: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x24dbbb811b0>}

In [15]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [7]:
def _preprocess_prediction_data(pred_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(pred_data,topology)

        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )
        data["cell_rxpwr_dbm"] = data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        data["relative_bearing"] = data.apply(
            lambda row: GISTools.get_relative_bearing(
                row["cell_az_deg"],
                row["cell_lat"],
                row["cell_lon"],
                row["latitude"],
                row["longitude"],
            ),
            axis=1,
        )
        return data

In [8]:
# Prediction
def _predictions(pred_data,topology,bayesian_digital_twins) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Predicts the received power for each User Equipment (UE) at different locations and ticks using Bayesian Digital Twins.
        It then determines the best cell for each UE to attach based on the predicted power values.
        """
        prediction_data = _preprocess_prediction_data(pred_data,topology)
        full_prediction_df = pd.DataFrame()

        # Loop over each 'tick'
        for tick, tick_df in prediction_data.groupby("tick"):
            # Loop over each 'cell_id' within the current 'tick'
            for cell_id, cell_df in tick_df.groupby("cell_id"):
                # Check if the Bayesian model for this cell_id exists
                if cell_id in bayesian_digital_twins:
                    # Perform the Bayesian prediction
                    pred_means_percell, _ = bayesian_digital_twins[
                        cell_id
                    ].predict_distributed_gpmodel(prediction_dfs=[cell_df])

                    # Assuming 'pred_means_percell' returns a list of predictions corresponding to the DataFrame index
                    cell_df["pred_means"] = pred_means_percell[0]

                    # Include additional necessary columns for the final DataFrame
                    cell_df["tick"] = tick
                    cell_df["cell_id"] = cell_id

                    # Append the predictions to the full DataFrame
                    full_prediction_df = pd.concat(
                        [full_prediction_df, cell_df], ignore_index=True
                    )
                else:
                    # Handle missing models, e.g., log a warning or initialize a default model
                    print(
                        f"No model available for cell_id {cell_id}, skipping prediction."
                    )

        full_prediction_df = full_prediction_df.rename(
            columns={"latitude": "loc_y", "longitude": "loc_x"}
        )
        predicted = perform_attachment(full_prediction_df, topology)

        return predicted, full_prediction_df

In [18]:
prediction_data = get_ue_data(params2)

In [19]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bayesian_digital_twins
)

Combined DataFrame:       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon  cell_id  \
0              0   48.510339 -16.462645     0     -90.0    -180.0        1   
1              0   48.510339 -16.462645     0       0.0       0.0        2   
2              0   48.510339 -16.462645     0      90.0     180.0        3   
3              1   19.286613  63.617111     0     -90.0    -180.0        1   
4              1   19.286613  63.617111     0       0.0       0.0        2   
...          ...         ...        ...   ...       ...       ...      ...   
4795          30  -64.060471  70.447505    49       0.0       0.0        2   
4796          30  -64.060471  70.447505    49      90.0     180.0        3   
4797          31 -152.742476  51.756565    49     -90.0    -180.0        1   
4798          31 -152.742476  51.756565    49       0.0       0.0        2   
4799          31 -152.742476  51.756565    49      90.0     180.0        3   

      cell_az_deg  cell_carrier_freq_mhz  


### Disect all the codes from MRO and put it to radp_library.py


In [9]:
bdt = {}

In [10]:
update_data = pd.read_csv('data/sim_data/combined_data_dump.csv')
update_data.head()

,latitude,longitude,cell_rxpower_dbm
0,59.806764,-22.625309,100.788266
1,59.806764,-22.625309,100.788266
2,59.806764,-22.625309,100.788266
3,54.857584,119.764151,99.608310
4,54.857584,119.764151,99.608310


## Attaching real **rxpower_dbm** data

In [11]:
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [12]:
real_rxdmb_data = pd.read_csv('data/sim_data/cell_rxpwr_debug.csv')
real_rxdmb_data.head()

,mock_ue_id,longitude,latitude,tick,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm,relative_bearing
0,5,-43.264737,-30.862211,20,-90.0,-180.0,cell_1,0,2100,15.700024,-99.812392,136.735263
1,10,54.978193,-55.259842,59,-90.0,-180.0,cell_1,0,2100,15.168050,-99.512981,234.978193
2,12,-98.676642,40.779019,56,-90.0,-180.0,cell_1,0,2100,16.493663,-100.240728,81.323358
3,11,5.606533,12.413489,36,-90.0,-180.0,cell_1,0,2100,16.249172,-100.111011,185.606533
4,14,-134.815655,-58.151045,87,-90.0,-180.0,cell_1,0,2100,15.081159,-99.463080,45.184345


In [13]:
update_data = real_rxdmb_data.copy()
update_data.drop(columns=['mock_ue_id', 'tick', 'cell_lat', 'cell_lon', 'cell_az_deg', 'cell_carrier_freq_mhz', 'log_distance', 'relative_bearing'], inplace=True)
# update_data = update_data[['']]
update_data.head()

,longitude,latitude,cell_id,cell_rxpwr_dbm
0,-43.264737,-30.862211,cell_1,-99.812392
1,54.978193,-55.259842,cell_1,-99.512981
2,-98.676642,40.779019,cell_1,-100.240728
3,5.606533,12.413489,cell_1,-100.111011
4,-134.815655,-58.151045,cell_1,-99.463080


In [14]:
def _preprocess_ue_update_data(update_data,topology) -> pd.DataFrame:
        data = _prepare_all_UEs_from_all_cells_df(update_data,topology)
        data["log_distance"] = data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        update_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))

        n_samples_update = []
        for df in update_per_cell_df:
            n_samples_update.append(df.shape[0])

        update_per_cell_df_processed = []
        for i in range(n_cell):
            update_per_cell_df_processed.append(
                get_percell_data(
                    data_in=update_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_update[i],
                )[0][0]
            )

        update_data = {}

        for i, df in enumerate(update_per_cell_df_processed):
            update_cell_id = idx_cell_id_mapping[i + 1]
            update_data[update_cell_id] = df

        for update_cell_id, update_data_idx in update_data.items():
            update_data_idx["cell_id"] = update_cell_id
            update_data_idx["cell_lat"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lat"].values[0]
            update_data_idx["cell_lon"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_lon"].values[0]
            update_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_az_deg"].values[0]
            update_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == update_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            update_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    update_data_idx["cell_az_deg"].values[0],
                    update_data_idx["cell_lat"].values[0],
                    update_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    update_data_idx["latitude"], update_data_idx["longitude"]
                )
            ]
        return update_data

In [ ]:
def train_or_update_rf_twin(new_data: pd.DataFrame,topology: pd.DataFrame, bayesian_digital_twins):
        try:
            if not isinstance(new_data, pd.DataFrame):
                raise TypeError("The input 'new_data' must be a pandas DataFrame.")
            
            new_data = new_data.rename(
                columns={"cell_rxpower_dbm": "cell_rxpwr_dbm"}
            )
            expected_columns = {"longitude", "latitude", "cell_rxpwr_dbm"}
            if not expected_columns.issubset(new_data.columns):
                raise ValueError(
                    f"The input DataFrame must contain the following columns: {expected_columns}"
                )

            if bayesian_digital_twins:
                update_data = new_data
                updated_data = _preprocess_ue_update_data(update_data, topology)
                updated_data_list = list(updated_data.values())
                print("Updated_List ",updated_data_list)

                for data_idx, update_data_df in enumerate(updated_data_list):
                    update_cell_id = data_idx + 1
                    if update_cell_id in bayesian_digital_twins:
                        print(f"Updating cell {update_cell_id} with {len(update_data_df)} samples.")
                        bayesian_digital_twins[
                            update_cell_id
                        ].update_trained_gpmodel([update_data_df])
            else:
                print(
                    "No Bayesian Digital Twins available for update. Training from scratch."
                )
                new_data = new_data.drop(
                    columns=["cell_rxpower_dbm"], errors="ignore"
                )
                _training(bayesian_digital_twins,maxiter=100, train_data=new_data,topology = topology)
        
            # return bayesian_digital_twins
        
        except TypeError as te:
            print(f"TypeError: {te}")
        except ValueError as ve:
            print(f"ValueError: {ve}")
        except KeyError as ke:
            print(f"KeyError: {ke}")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

In [16]:
bdt

{}

In [17]:
updates = train_or_update_rf_twin(update_data,topology,bdt)

No Bayesian Digital Twins available for update. Training from scratch.
Combined DataFrame:        longitude   latitude  cell_rxpwr_dbm  cell_lat  cell_lon  cell_id  \
0     -43.264737 -30.862211      -99.812392     -90.0    -180.0        1   
1     -43.264737 -30.862211      -99.812392       0.0       0.0        2   
2     -43.264737 -30.862211      -99.812392      90.0     180.0        3   
3      54.978193 -55.259842      -99.512981     -90.0    -180.0        1   
4      54.978193 -55.259842      -99.512981       0.0       0.0        2   
...          ...        ...             ...       ...       ...      ...   
17995  89.254786 -82.596233     -100.385621       0.0       0.0        2   
17996  89.254786 -82.596233     -100.385621      90.0     180.0        3   
17997  20.088340 -55.786882     -100.297751     -90.0    -180.0        1   
17998  20.088340 -55.786882     -100.297751       0.0       0.0        2   
17999  20.088340 -55.786882     -100.297751      90.0     180.0        3 

[2025-05-07 16:53:40,223] INFO:  Iter 1/100 - Loss: 0.758 (delta=inf)
[2025-05-07 16:53:41,271] INFO:  Iter 2/100 - Loss: 0.739 (delta=-0.018517)
[2025-05-07 16:53:42,167] INFO:  Iter 3/100 - Loss: 0.721 (delta=-0.018668)
[2025-05-07 16:53:43,156] INFO:  Iter 4/100 - Loss: 0.702 (delta=-0.018836)
[2025-05-07 16:53:44,335] INFO:  Iter 5/100 - Loss: 0.683 (delta=-0.019023)
[2025-05-07 16:53:44,887] INFO:  Iter 6/100 - Loss: 0.664 (delta=-0.019215)
[2025-05-07 16:53:45,702] INFO:  Iter 7/100 - Loss: 0.644 (delta=-0.019419)
[2025-05-07 16:53:46,224] INFO:  Iter 8/100 - Loss: 0.625 (delta=-0.019624)
[2025-05-07 16:53:46,934] INFO:  Iter 9/100 - Loss: 0.605 (delta=-0.019817)
[2025-05-07 16:53:47,475] INFO:  Iter 10/100 - Loss: 0.585 (delta=-0.020020)
[2025-05-07 16:53:47,931] INFO:  Iter 11/100 - Loss: 0.565 (delta=-0.020228)
[2025-05-07 16:53:48,440] INFO:  Iter 12/100 - Loss: 0.544 (delta=-0.020430)
[2025-05-07 16:53:48,989] INFO:  Iter 13/100 - Loss: 0.523 (delta=-0.020638)
[2025-05-07 16

In [18]:
bdt

{1: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d3aa3d5e10>,
 2: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d3fe758190>,
 3: <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d3fe758d00>}

In [19]:
params2 = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 7,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 10,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.8,
                    "variance": 0.5,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [20]:
prediction_data = get_ue_data(params2)

In [21]:
prediction_data.head()

,mock_ue_id,lon,lat,tick
0,0,48.510339,-16.462645,0
1,1,19.286613,63.617111,0
2,2,21.315702,-47.889952,0
3,3,-70.559364,-79.511709,0
4,4,-168.916011,-39.338640,0


In [22]:
predictions, full_prediction_df = _predictions(
    pred_data=prediction_data,
    topology=topology,
    bayesian_digital_twins = bdt
)

Combined DataFrame:       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon  cell_id  \
0              0   48.510339 -16.462645     0     -90.0    -180.0        1   
1              0   48.510339 -16.462645     0       0.0       0.0        2   
2              0   48.510339 -16.462645     0      90.0     180.0        3   
3              1   19.286613  63.617111     0     -90.0    -180.0        1   
4              1   19.286613  63.617111     0       0.0       0.0        2   
...          ...         ...        ...   ...       ...       ...      ...   
4795          30  -64.060471  70.447505    49       0.0       0.0        2   
4796          30  -64.060471  70.447505    49      90.0     180.0        3   
4797          31 -152.742476  51.756565    49     -90.0    -180.0        1   
4798          31 -152.742476  51.756565    49       0.0       0.0        2   
4799          31 -152.742476  51.756565    49      90.0     180.0        3   

      cell_az_deg  cell_carrier_freq_mhz  


c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance va

In [24]:
predictions.head()

,loc_x,loc_y,cell_id,sinr_db,rsrp_dbm
0,48.510339,-16.462645,2.0,-2.701067,-102.223811
1,19.286613,63.617111,3.0,-2.269515,-101.852432
2,21.315702,-47.889952,1.0,-2.645710,-102.127219
3,-70.559364,-79.511709,1.0,-1.630991,-101.313783
4,-168.916011,-39.338640,1.0,-2.507067,-102.250907


In [ ]:
full_prediction_df.head()

# cell_rxpwr_dbm --> calculated
# rxpower_dbm --> predicted

,mock_ue_id,loc_x,loc_y,tick,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm,relative_bearing,rxpower_dbm,rxpower_stddev_dbm,pred_means
0,0,48.510339,-16.462645,0,-90.0,-180.0,1,0,2800,15.917947,-102.430902,228.510339,-102.436820,0.032373,-102.436820
1,1,19.286613,63.617111,0,-90.0,-180.0,1,0,2800,16.654617,-102.823854,199.286613,-102.834308,0.032368,-102.834308
2,2,21.315702,-47.889952,0,-90.0,-180.0,1,0,2800,15.360440,-102.121234,201.315702,-102.127219,0.032362,-102.127219
3,3,-70.559364,-79.511709,0,-90.0,-180.0,1,0,2800,13.970414,-101.297346,109.440636,-101.313783,0.032955,-101.313783
4,4,-168.916011,-39.338640,0,-90.0,-180.0,1,0,2800,15.545318,-102.225153,11.083989,-102.250907,0.033513,-102.250907


In [27]:
# Training from scratch done now will update the exisitng BDT
update_not_from_scratch = train_or_update_rf_twin(update_data,topology,bdt)

Combined DataFrame:        longitude   latitude  cell_rxpwr_dbm  cell_lat  cell_lon  cell_id  \
0     -43.264737 -30.862211      -99.812392     -90.0    -180.0        1   
1     -43.264737 -30.862211      -99.812392       0.0       0.0        2   
2     -43.264737 -30.862211      -99.812392      90.0     180.0        3   
3      54.978193 -55.259842      -99.512981     -90.0    -180.0        1   
4      54.978193 -55.259842      -99.512981       0.0       0.0        2   
...          ...        ...             ...       ...       ...      ...   
17995  89.254786 -82.596233     -100.385621       0.0       0.0        2   
17996  89.254786 -82.596233     -100.385621      90.0     180.0        3   
17997  20.088340 -55.786882     -100.297751     -90.0    -180.0        1   
17998  20.088340 -55.786882     -100.297751       0.0       0.0        2   
17999  20.088340 -55.786882     -100.297751      90.0     180.0        3   

       cell_az_deg  cell_carrier_freq_mhz  
0                0     

c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\linear_operator\utils\cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-06 to the diagonal
  warnings.warn(
c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\linear_operator\utils\cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-05 to the diagonal
  warnings.warn(
c:\Users\rafid\OneDrive\Desktop\VSCode\maveric\.venv\lib\site-packages\linear_operator\utils\cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-04 to the diagonal
  warnings.warn(


An unexpected error occurred: Matrix not positive definite after repeatedly adding jitter up to 1.0e-04.
